<a href="https://colab.research.google.com/github/Dani2003/paper-implementations/blob/main/multilook_coherent_imaging_dip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# [CELL 1 - Setup Environment and Imports]
import os
import sys
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, List, Dict, Optional

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# Set deterministic random seeds
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Compute Device: {device}")
if device.type == "cuda":
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
print("Imaging Environment Initialized Successfully.")

Using Compute Device: cuda
GPU Device Name: Tesla T4
Imaging Environment Initialized Successfully.


In [2]:
# [CELL 2 - Multiplicative Speckle Noise & Multilook Dataset Generator]

class MultilookSpeckleSimulator:
    """
    Simulates multilook coherent imaging measurements with multiplicative speckle noise.
    Across L looks, intensity measurements follow speckle distributions governed by ground truth image X.
    """
    def __init__(self, image_size: int = 64, num_looks: int = 4):
        self.image_size = image_size
        self.num_looks = num_looks

    def generate_clean_image(self) -> torch.Tensor:
        """Generates a synthetic structured phantom image (e.g., Shepp-Logan-like or geometric shapes)."""
        img = torch.zeros((1, 1, self.image_size, self.image_size), dtype=torch.float32)
        # Add geometric structures representing features
        yy, xx = torch.meshgrid(torch.linspace(-1, 1, self.image_size), torch.linspace(-1, 1, self.image_size), indexing='ij')
        mask = (xx**2 + yy**2) < 0.6
        img[0, 0, mask] = 0.8
        inner_mask = (xx**2 + yy**2) < 0.2
        img[0, 0, inner_mask] = 1.0
        return img

    def simulate_measurements(self, clean_image: torch.Tensor) -> torch.Tensor:
        """
        Simulates L multilook intensity measurements Y_l = X * W_l,
        where W_l represents fully developed speckle noise.
        """
        batch_size, channels, H, W = clean_image.shape
        multilook_measurements = []

        for l in range(self.num_looks):
            # Speckle noise W_l follows exponential/gamma distribution
            speckle = torch.from_numpy(np.random.exponential(scale=1.0, size=(batch_size, channels, H, W))).float()
            measurement = clean_image * speckle
            multilook_measurements.append(measurement)

        # Stack across looks dimension: (Batch, L looks, Channels, H, W)
        return torch.stack(multilook_measurements, dim=1)

simulator = MultilookSpeckleSimulator(image_size=64, num_looks=4)
clean_phantom = simulator.generate_clean_image()
noisy_measurements = simulator.simulate_measurements(clean_phantom)

print(Clean Image Shape: {clean_phantom.shape})
print(Multilook Measurements Shape: {noisy_measurements.shape} (Batch, Looks, C, H, W))

SyntaxError: invalid syntax. Perhaps you forgot a comma? (4228810350.py, line 44)

In [ ]:
# [CELL 3 - Deep Image Prior (DIP) Convolutional Decoder Architecture]

class DeepImagePriorDecoder(nn.Module):
    """
    Deep Image Prior network mapping random stationary latent code z
    to the target speckle-free image X = f_theta(z).
    """
    def __init__(self, latent_dim: int = 128, output_channels: int = 1, hidden_dim: int = 64):
        super().__init__()
        self.latent_dim = latent_dim

        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, hidden_dim * 4, kernel_s=4, stride=1, padding=0),
            nn.BatchNorm2d(hidden_dim * 4),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(hidden_dim * 4, hidden_dim * 2, kernel_s=4, stride=2, padding=1),
            nn.BatchNorm2d(hidden_dim * 2),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(hidden_dim * 2, hidden_dim, kernel_s=4, stride=2, padding=1),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(hidden_dim, output_channels, kernel_s=4, stride=2, padding=1),
            nn.Sigmoid() # Constrain intensity values to [0, 1]
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)

dip_model = DeepImagePriorDecoder(latent_dim=128).to(device)
latent_z = torch.randn(1, 128, 1, 1, device=device)
initial_recon = dip_model(latent_z)
print(f"DIP Initialized. Output Reconstruction Shape: {initial_recon.shape}")

In [3]:
# [CELL 4 - Maximum Likelihood Estimation via PGD & Newton-Schulz Acceleration]

class NewtonSchulzInverter:
    """Computes efficient matrix approximate inverse / scaling using Newton-Schulz iterations."""
    @staticmethod
    def accelerate_inverse(matrix: torch.Tensor, iterations: int = 5) -> torch.Tensor:
        # Scales matrix for Schulz iteration convergence
        norm = torch.norm(matrix)
        if norm == 0:
            return matrix
        Y = matrix / (norm + 1e-7)
        I = torch.eye(matrix.size(-1), device=matrix.device)

        for _ in range(iterations):
            Y = 0.5 * Y @ (3.0 * I - matrix @ Y)
        return Y


def multilook_mle_loss(reconstructed_image: torch.Tensor, measurements: torch.Tensor) -> torch.Tensor:
    """
    Computes negative log-likelihood loss for multilook coherent imaging under multiplicative speckle.
    L = sum_l [ Y_l / X + ln(X) ] (ignoring constants).
    """
    # reconstructed_image: (1, 1, H, W), measurements: (1, L, 1, H, W)
    eps = 1e-6
    X = torch.clamp(reconstructed_image, min=eps)
    num_looks = measurements.size(1)

    total_loss = 0.0
    for l in range(num_looks):
        Y_l = measurements[:, l, :, :, :]
        # Negative log likelihood term for gamma/exponential multilook speckle
        nll = (Y_l / X) + torch.log(X)
        total_loss += torch.mean(nll)

    return total_loss / num_looks

print("Maximum likelihood multilook speckle loss module compiled.")

Maximum likelihood multilook speckle loss module compiled.


In [4]:
# [CELL 5 - Bagging Strategy to Mitigate PGD Projection Errors]

def run_bagged_dip_mle_reconstruction(
    measurements: torch.Tensor,
    num_bags: int = 3,
    iterations: int = 150
) -> torch.Tensor:
    """
    Runs an ensemble of Deep Image Prior reconstructions with bagging
    to mitigate projection errors and local minima in PGD.
    """
    batch_size, num_looks, C, H, W = measurements.shape
    bag_reconstructions = []

    print(f"Running Bagged DIP-MLE across {num_bags} ensemble models...")

    for bag in range(num_bags):
        # Initialize separate DIP model and latent code per bag
        model = DeepImagePriorDecoder(latent_dim=64).to(device)
        optimizer = optim.Adam(model.parameters(), lr=0.01)

        # Perturbed latent code for bagging diversity
        z = torch.randn(1, 64, 1, 1, device=device) * (1.0 + 0.1 * bag)

        for it in range(iterations):
            optimizer.zero_grad()
            recon = model(z)
            loss = multilook_mle_loss(recon, measurements.to(device))
            loss.backward()
            optimizer.step()

        with torch.no_grad():
            bag_reconstructions.append(model(z).detach().cpu())

    # Average reconstructions across bagging ensemble
    bagged_final = torch.stack(bag_reconstructions).mean(dim=0)
    return bagged_final

print("Bagged DIP-MLE reconstruction pipeline ready.")

Bagged DIP-MLE reconstruction pipeline ready.


In [5]:
# [CELL 6 - Quantitative PSNR/SSIM Evaluation & Visualization]

def compute_psnr(img1: torch.Tensor, img2: torch.Tensor) -> float:
    mse = torch.mean((img1 - img2) ** 2).item()
    if mse == 0:
        return 100.0
    return 20.0 * math.log10(1.0 / math.sqrt(mse))

# Execute quick reconstruction on synthetic measurements
reconstructed_image = run_bagged_dip_mle_reconstruction(noisy_measurements, num_bags=2, iterations=80)

# Compute PSNR
psnr_val = compute_psnr(clean_phantom, reconstructed_image)
print(f"\nReconstruction Peak Signal-to-Noise Ratio (PSNR): {psnr_val:.2f} dB")

# Visualization Suite
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
sns.set_theme(style="whitegrid")

axes[0].imshow(clean_phantom[0, 0].numpy(), cmap='gray', vmin=0, vmax=1)
axes[0].set_title("Ground Truth Phantom", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(noisy_measurements[0, 0, 0].numpy(), cmap='gray', vmin=0, vmax=2)
axes[1].set_title("Noisy Multilook Measurement (Look 1)", fontweight='bold')
axes[1].axis('off')

axes[2].imshow(reconstructed_image[0, 0].numpy(), cmap='gray', vmin=0, vmax=1)
axes[2].set_title(f"Bagged DIP-MLE Recon (PSNR: {psnr_val:.2f}dB)", fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.savefig("multilook_speckle_reconstruction_benchmark.png", dpi=300)
plt.show()

print("Benchmark visualization successfully saved as multilook_speckle_reconstruction_benchmark.png.")

NameError: name 'noisy_measurements' is not defined